# Alzheimer's Disease

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pydicom as dicom
import sys
import matplotlib.path as mplPath
import pandas as pd
import shutil
import os
import glob
import csv
from skimage import io

import torch
from torchvision import datasets
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision.io import read_image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader,WeightedRandomSampler

In [ ]:
# Train-Test Split
import splitfolders

In [ ]:
# defining transformation
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
     transforms.RandomRotation(30),
    #transforms.GaussianBlur(kernel_size = (3, 3), sigma = (0.3, 0.7)),
     #transforms.RandomVerticalFlip(),
     transforms.Resize((224, 224)),
    transforms.ToTensor()])

In [ ]:
test_transforms = transforms.Compose(
    [transforms.Resize((224, 224)),
transforms.ToTensor()])
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
trainset = datasets.ImageFolder('Splitted2D/train', transform=train_transforms)
valset = datasets.ImageFolder('Splitted2D/val', transform = val_transforms)
testset = datasets.ImageFolder('Splitted2D/test', transform=test_transforms)

In [ ]:
# Sampling
trainset.class_to_idx

In [ ]:
idx2class = {v: k for k, v in trainset.class_to_idx.items()}
idx2class

In [ ]:
def get_class_distribution(dataset_obj):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for element in dataset_obj:
        y_lbl = element[1]
        y_lbl = idx2class[y_lbl]
        count_dict[y_lbl] += 1
            
    return count_dict
print("Distribution of classes: \n", get_class_distribution(trainset))


In [ ]:
target_list = torch.tensor(trainset.targets)
class_count = [i for i in get_class_distribution(trainset).values()]
class_weights = 1./torch.tensor(class_count, dtype=torch.float) 
class_weights


In [ ]:
print(class_count)

In [ ]:
class_weights_all = class_weights[target_list]
class_weights_all

In [ ]:
weighted_sampler = WeightedRandomSampler(
    weights=class_weights_all,
    num_samples=len(class_weights_all),
    replacement=True
)

In [ ]:
#dataloaders
BATCH_SIZE = 32
train_dataloader = DataLoader(trainset, batch_size=BATCH_SIZE, sampler=weighted_sampler)
val_loader = DataLoader(valset, batch_size = BATCH_SIZE, shuffle = True)
test_dataloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
#testing shape
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0][1].squeeze()
label = train_labels[0]
plt.imshow(img, cmap='gray')
plt.show()
print(f"Label: {label}")

In [ ]:
for i in img:
    print(i)

In [ ]:
#testing shape
test_features, test_labels = next(iter(test_dataloader))
print(f"Feature batch shape: {test_features.size()}")
print(f"Labels batch shape: {test_labels.size()}")
img = test_features[0][1].squeeze()
label = test_labels[0]
plt.imshow(img, cmap='gray')
plt.show()
print(f"Label: {label}")

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
from torchsummary import summary

In [ ]:
from torchvision.models import googlenet, GoogLeNet_Weights
from torchsummary import summary

https://discuss.pytorch.org/t/load-only-a-part-of-the-network-with-pretrained-weights/88397

https://towardsdatascience.com/three-ways-to-build-a-neural-network-in-pytorch-8cea49f9a61a

https://pytorch.org/tutorials/beginner/finetuning_torchvision_models_tutorial.html#create-the-optimizer

https://medium.com/analytics-vidhya/pytorch-for-deep-learning-binary-classification-logistic-regression-382abd97fb43

https://pytorch.org/vision/stable/models.html

In [ ]:
from torchvision import models
num_epochs = 40
lr = 0.01
num_class = 4
model = models.resnet18(weights='IMAGENET1K_V1')

In [ ]:
summary(model, (3, 224, 224))

In [ ]:
model.train()

In [ ]:
count = 0
# for child in model.children():
#     #print(count, child)
#     count += 1
#     if (count < 9):
#         for param in child.parameters():
#             param.requires_grad = False
# model.fc = nn.Sequential(
#     nn.Dropout(0.3),
#     nn.Linear(2048, 4)
#)
model.fc = nn.Sequential(
    nn.Dropout(0.2, inplace = False),
    nn.Linear(256, 4),
    nn.Softmax(dim = -1)
)
    

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr= 0.01) 
model = model.to(device)

In [ ]:
def train_loop(train_loader, val_loader, model, loss_fn, optimizer):
    size = len(train_loader.dataset)
    correct = 0
    samples = 0
    
    for batch, (X, y) in enumerate(train_loader):
        # Compute prediction and loss
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
#         for name, param in model.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                print(gradient_norm)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            samples += pred.size(0)
            if batch % 5 ==0:
                print(f"Acc: {float(correct) / float(samples) * 100:.2f}% Out of {samples} \n")
                correct = 0
                samples = 0

        if batch % 5 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]  \n")
    val_correct, val_samples = 0, 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        with torch.no_grad():
            val_correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            val_samples += pred.size(0)
    print(f"validation accuracy: {val_correct / val_samples * 100:.2f}%")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}% out of {size}, Avg loss: {test_loss:>8f} \n")

In [ ]:
num_epochs = 50
for t in range(num_epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, val_loader, model, loss, optimizer)


In [ ]:
import nbformat

with open("kevin-resnet50.ipynb", "r") as file:
    nb_corrupted = nbformat.reader.read(file)

nbformat.validator.validate(nb_corrupted)
# <stdin>:1: MissingIDFieldWarning: Code cell is missing an id field, 
# this will become a hard error in future nbformat versions. 
# You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). 
# Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.

nb_fixed = nbformat.validator.normalize(nb_corrupted)
nbformat.validator.validate(nb_fixed[1])
# Produces no warnings or errors.

with open("fixed_notebook.ipynb", "w") as file:
    nbformat.write(nb_fixed[1], file)


In [ ]:
test_loop(test_dataloader, model, loss)

# resnext50:
#Test Error: 40 epochs
#Accuracy: 82.2% out of 73, Avg loss: 0.964113 

# resnet50:
# Test Error: 49 epochs
# Accuracy: 86.3% out of 73, Avg loss: 0.607938 
